<a href="https://colab.research.google.com/github/asaveraasad-data/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [28]:
import os
import subprocess

REPO_URL = "https://github.com/asaveraasad-data/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)

os.chdir(REPO_DIR)

print("Current directory:", os.getcwd())

Current directory: /content/flyrank-ml-internship/flyrank-ml-internship


In [29]:
import os

print(os.path.exists("data/raw/content_refresh_anonymized.csv"))

True


# 1. Ranked Actions + Reason Codes

## Objective

This section converts the validated model output into a ranked editorial action queue. Rather than updating content automatically, the model identifies pages that appear to be the strongest candidates for review based on the observed patterns in the data.

The ranked queue is intended to support content editors by prioritizing pages according to their predicted refresh opportunity, confidence level, and supporting reason codes.

The recommendations should be interpreted as decision-support rather than guaranteed outcomes. Every recommended action should be reviewed by a human editor before implementation.

In [30]:
from pathlib import Path
import pandas as pd

# -------------------------------------------------------
# Project Root
# -------------------------------------------------------
PROJECT_ROOT = Path("/content/flyrank-ml-internship")

RAW_DATA = PROJECT_ROOT / "data" / "raw"

# Existing model outputs
INPUT_OUTPUT_DIR = PROJECT_ROOT / "outputs"

# Assignment output folder
EXPORT_DIR = PROJECT_ROOT / "work" / "outputs"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------
# Load datasets
# -------------------------------------------------------

raw_data = pd.read_csv(
    RAW_DATA / "content_refresh_anonymized.csv"
)

queue = pd.read_csv(
    INPUT_OUTPUT_DIR / "refresh_queue_sample.csv"
)

print(f"Raw Dataset Shape : {raw_data.shape}")
print(f"Refresh Queue Shape : {queue.shape}")

display(queue.head())

Raw Dataset Shape : (30000, 44)
Refresh Queue Shape : (200, 28)


,final_rank,content_id,client_id,final_refresh_score,best_model_name,best_model_probability,baseline_refresh_score,confidence,suggested_action,final_reason_codes,...,word_count,trend_direction,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier
0,1,content_1f080331fa2b,client_3fdba35f04,81.636697,random_forest,0.782079,0.844481,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,...,1404.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,good,page_1
1,2,content_6aa43079fb0c,client_3fdba35f04,81.447656,random_forest,0.788105,0.825477,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1457.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1
2,3,content_d6570c51c9bd,client_3fdba35f04,81.430346,random_forest,0.847372,0.695884,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1362.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,moderate,striking
3,4,content_72e800a9c214,client_3fdba35f04,81.034960,random_forest,0.774371,0.842545,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1371.0,down,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,good,page_1
4,5,content_e04eb9549989,client_3fdba35f04,80.873188,random_forest,0.814805,0.749468,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1408.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1


In [31]:
# -------------------------------------------------------
# Dataset Overview
# -------------------------------------------------------

print("=" * 60)
print("RAW DATASET")
print("=" * 60)

print(f"Number of content pages : {len(raw_data):,}")
print(f"Number of features      : {raw_data.shape[1]}")

print("\nContent Types")
display(raw_data["content_type"].value_counts())

print("\nTrend Direction")
display(raw_data["trend_direction"].value_counts())

print("\n")

print("=" * 60)
print("MODEL OUTPUT")
print("=" * 60)

print(f"Recommended Pages : {len(queue):,}")

display(queue.head())

RAW DATASET
Number of content pages : 30,000
Number of features      : 44

Content Types


,count
content_type,
keyword article,27207
feedly article,2096
comparison article,697



Trend Direction


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152




MODEL OUTPUT
Recommended Pages : 200


,final_rank,content_id,client_id,final_refresh_score,best_model_name,best_model_probability,baseline_refresh_score,confidence,suggested_action,final_reason_codes,...,word_count,trend_direction,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier
0,1,content_1f080331fa2b,client_3fdba35f04,81.636697,random_forest,0.782079,0.844481,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,...,1404.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,good,page_1
1,2,content_6aa43079fb0c,client_3fdba35f04,81.447656,random_forest,0.788105,0.825477,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1457.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1
2,3,content_d6570c51c9bd,client_3fdba35f04,81.430346,random_forest,0.847372,0.695884,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1362.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,moderate,striking
3,4,content_72e800a9c214,client_3fdba35f04,81.034960,random_forest,0.774371,0.842545,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1371.0,down,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,good,page_1
4,5,content_e04eb9549989,client_3fdba35f04,80.873188,random_forest,0.814805,0.749468,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1408.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1


### Dataset Summary

The original dataset contains historical content performance metrics collected over a trailing observation window. The validated model converts these historical signals into a ranked recommendation queue.

The action queue is the primary input for this playbook because it represents the final output of the validated machine learning pipeline developed in previous stages of the project.

In [32]:
# Top priority recommendations

editorial_queue = (
    queue[
        [
            "final_rank",
            "content_id",
            "suggested_action",
            "confidence",
            "best_model_probability",
            "final_refresh_score",
            "final_reason_codes"
        ]
    ]
    .sort_values("final_rank")
)

display(editorial_queue.head(20))

,final_rank,content_id,suggested_action,confidence,best_model_probability,final_refresh_score,final_reason_codes
0,1,content_1f080331fa2b,refresh_and_review_ctr,high,0.782079,81.636697,declining_with_demand|low_ctr_visible_page|low...
1,2,content_6aa43079fb0c,refresh_and_review_ctr,high,0.788105,81.447656,declining_with_demand|low_ctr_visible_page|mod...
2,3,content_d6570c51c9bd,refresh_and_review_ctr,medium,0.847372,81.430346,declining_with_demand|low_ctr_visible_page|mod...
3,4,content_72e800a9c214,refresh_and_review_ctr,high,0.774371,81.034960,declining_with_demand|low_ctr_visible_page|mod...
4,5,content_e04eb9549989,refresh_and_review_ctr,medium,0.814805,80.873188,declining_with_demand|low_ctr_visible_page|mod...
5,6,content_b69288c5e701,refresh_and_review_ctr,high,0.795713,80.754770,declining_with_demand|low_ctr_visible_page|mod...
6,7,content_9b6df29f7889,refresh_and_review_ctr,high,0.846245,80.632923,declining_with_demand|low_ctr_visible_page|mod...
7,8,content_bb6ebb5ec8c8,refresh_and_review_ctr,high,0.834638,80.371236,declining_with_demand|low_ctr_visible_page|mod...
8,9,content_4d76cdb3387b,refresh_and_review_ctr,medium,0.843092,80.362748,declining_with_demand|low_ctr_visible_page|mod...
9,10,content_b4f35d640b1c,refresh,medium,0.843803,80.321757,declining_with_demand|model_decline_risk|visib...


In [33]:
action_summary = (
    queue
    .groupby("suggested_action")
    .agg(
        Pages=("content_id","count"),
        Average_Score=("final_refresh_score","mean"),
        Average_Confidence=("best_model_probability","mean")
    )
    .round(3)
)

display(action_summary)

,Pages,Average_Score,Average_Confidence
suggested_action,,,
refresh,35,77.879,0.826
refresh_and_review_ctr,130,78.030,0.791
refresh_and_review_engagement,35,77.717,0.817


In [34]:
reason_summary = (
    queue["final_reason_codes"]
    .value_counts()
    .reset_index()
)

reason_summary.columns = [
    "Reason Code",
    "Frequency"
]

display(reason_summary)

,Reason Code,Frequency
0,declining_with_demand|low_ctr_visible_page|mod...,74
1,declining_with_demand|model_decline_risk|visib...,35
2,declining_with_demand|low_ctr_visible_page|low...,33
3,declining_with_demand|low_engagement_visible_p...,33
4,declining_with_demand|page_one_decay_risk|low_...,11
5,declining_with_demand|page_one_decay_risk|low_...,9
6,low_ctr_visible_page|model_decline_risk|visibl...,2
7,low_ctr_visible_page|low_engagement_visible_pa...,1
8,declining_with_demand|page_one_decay_risk|low_...,1
9,stale_visible_page|declining_with_demand|low_e...,1


In [35]:
def priority_bucket(rank):

    if rank <= 50:
        return "Immediate"

    elif rank <= 200:
        return "High"

    elif rank <= 500:
        return "Medium"

    else:
        return "Monitor"


queue["Priority"] = queue["final_rank"].apply(priority_bucket)

display(
    queue[
        [
            "Priority",
            "suggested_action",
            "confidence",
            "final_reason_codes"
        ]
    ].head(20)
)

,Priority,suggested_action,confidence,final_reason_codes
0,Immediate,refresh_and_review_ctr,high,declining_with_demand|low_ctr_visible_page|low...
1,Immediate,refresh_and_review_ctr,high,declining_with_demand|low_ctr_visible_page|mod...
2,Immediate,refresh_and_review_ctr,medium,declining_with_demand|low_ctr_visible_page|mod...
3,Immediate,refresh_and_review_ctr,high,declining_with_demand|low_ctr_visible_page|mod...
4,Immediate,refresh_and_review_ctr,medium,declining_with_demand|low_ctr_visible_page|mod...
5,Immediate,refresh_and_review_ctr,high,declining_with_demand|low_ctr_visible_page|mod...
6,Immediate,refresh_and_review_ctr,high,declining_with_demand|low_ctr_visible_page|mod...
7,Immediate,refresh_and_review_ctr,high,declining_with_demand|low_ctr_visible_page|mod...
8,Immediate,refresh_and_review_ctr,medium,declining_with_demand|low_ctr_visible_page|mod...
9,Immediate,refresh,medium,declining_with_demand|model_decline_risk|visib...


### Interpretation

The validated model produces a ranked recommendation queue that supports editorial prioritization. Higher-ranked pages are associated with stronger model scores and confidence levels. These recommendations are intended to guide human review rather than automate publishing decisions. Final content changes should always be evaluated by editors before implementation.

## 2. Intended Use and Limits

### Intended Use

The content action playbook is designed to support editorial teams by prioritizing pages for manual review. It converts validated machine learning predictions into a ranked list of recommended actions, helping teams allocate limited content resources more efficiently.

The recommendations are intended for:
- Prioritizing content refresh efforts.
- Identifying pages that may benefit from editorial updates.
- Supporting SEO and content strategy discussions.
- Planning review schedules based on predicted opportunity.

The model provides **decision-support**, not automated decisions. All recommendations should be reviewed by a human editor before implementation.

### Limits

This playbook has several important limitations:

- The model was trained on historical observations and does not guarantee future search performance.
- Search engine algorithms and user behavior change over time, which may reduce model effectiveness.
- Business priorities, seasonal trends, and editorial considerations are not captured by the model.
- High-ranked pages should not automatically be refreshed without verifying factual accuracy and search intent.
- The recommendations should be interpreted as guidance rather than certainty.

In [36]:
# -------------------------------------------------------
# Summary of Recommendation Queue
# -------------------------------------------------------

summary = {
    "Total Recommended Pages": len(queue),
    "Unique Suggested Actions": queue["suggested_action"].nunique(),
    "Average Refresh Score": round(queue["final_refresh_score"].mean(), 3),
    "Average Model Probability": round(queue["best_model_probability"].mean(), 3),
    "High Confidence Recommendations": (queue["confidence"] == "High").sum()
}

summary_df = pd.DataFrame(summary.items(), columns=["Metric", "Value"])

display(summary_df)

,Metric,Value
0,Total Recommended Pages,200.000
1,Unique Suggested Actions,3.000
2,Average Refresh Score,77.949
3,Average Model Probability,0.802
4,High Confidence Recommendations,0.000


### Interpretation

The recommendation queue is intended to help editors focus their attention on higher-priority pages while maintaining human oversight throughout the review process.

The model ranks pages according to historical patterns observed during training. These rankings should be used as an input to editorial decision-making rather than as automatic publishing instructions. Human judgement remains essential for evaluating content quality, business relevance, and factual accuracy before any changes are made.

## 3. Human Review + the No-Go List

### Human Review

The model provides a ranked list of pages that may deserve editorial attention. Before implementing any recommendation, a human reviewer should verify that the suggested action aligns with business goals, user needs, and content quality standards.

Recommended review checklist:

- Verify factual accuracy and ensure information is up to date.
- Confirm that the page still matches the intended search intent.
- Check for outdated statistics, screenshots, or references.
- Ensure the content follows current editorial and brand guidelines.
- Review internal and external links for broken or outdated references.
- Consider seasonal relevance or recent business changes before updating content.

### No-Go List

The following tasks should **not** be automated using this model:

- Automatically publishing updated content.
- Deleting or redirecting pages solely based on the model's recommendation.
- Making legal, financial, or medical content changes without expert review.
- Replacing editorial judgement with model predictions.
- Assuming that refreshing a page will improve rankings or traffic.
- Ignoring business priorities or compliance requirements.

In [37]:
# -------------------------------------------------------
# Human Review Summary
# -------------------------------------------------------

review_summary = (
    queue.groupby(["suggested_action", "confidence"])
         .size()
         .reset_index(name="Number of Pages")
         .sort_values("Number of Pages", ascending=False)
)

display(review_summary)

,suggested_action,confidence,Number of Pages
2,refresh_and_review_ctr,high,102
4,refresh_and_review_engagement,high,35
3,refresh_and_review_ctr,medium,28
0,refresh,high,24
1,refresh,medium,11


### Interpretation

The recommendation queue should be treated as a prioritization tool rather than an automated decision system. Higher-confidence recommendations may deserve earlier review, but every suggested action requires human verification before implementation.

Editorial expertise remains essential for evaluating content quality, business priorities, compliance requirements, and user intent. The model assists with prioritization but does not replace professional judgement.

## 4. Monitoring / Retrain Triggers

### Monitoring Strategy

The recommendations generated by this playbook should be monitored periodically to ensure they remain useful as search behavior and content performance evolve.

The model should be reviewed when noticeable changes occur in the underlying data or when recommendation quality appears to decline.

Recommended monitoring activities include:

- Compare recent content performance against historical trends.
- Track whether recommended pages continue to match editorial priorities.
- Review the distribution of suggested actions over time.
- Monitor prediction confidence for unexpected shifts.
- Check whether important content features change substantially from the training data.

### Suggested Retraining Triggers

Retraining should be considered when one or more of the following conditions occur:

- A sustained decline in model performance during validation.
- Significant changes in search trends or user behavior.
- Large increases in newly published or updated content.
- Changes to the available feature set.
- Scheduled model maintenance every 3–6 months when sufficient new data becomes available.

These triggers are recommendations rather than fixed operational requirements because this project represents a decision-support prototype rather than a production deployment.

In [38]:
# -------------------------------------------------------
# Monitoring Dashboard
# -------------------------------------------------------

monitoring_summary = pd.DataFrame({
    "Metric": [
        "Total Pages",
        "Average Refresh Score",
        "Average Model Probability",
        "High Confidence",
        "Medium Confidence",
        "Low Confidence"
    ],
    "Current Value": [
        len(queue),
        round(queue["final_refresh_score"].mean(), 3),
        round(queue["best_model_probability"].mean(), 3),
        (queue["confidence"] == "High").sum(),
        (queue["confidence"] == "Medium").sum(),
        (queue["confidence"] == "Low").sum()
    ]
})

display(monitoring_summary)

,Metric,Current Value
0,Total Pages,200.000
1,Average Refresh Score,77.949
2,Average Model Probability,0.802
3,High Confidence,0.000
4,Medium Confidence,0.000
5,Low Confidence,0.000


### Interpretation

Monitoring helps ensure that the recommendation queue remains aligned with current content performance. A noticeable change in confidence distribution, feature characteristics, or editorial outcomes may indicate that the model should be reviewed or retrained.

For this project, retraining is presented as a periodic maintenance activity rather than an automated process. Human evaluation should always precede changes to the model or its recommendations.

## 5. Exports for the Paper

### Objective

This section exports the final editorial recommendation queue so it can be reused in the research paper. The exported files provide a reproducible record of the model's recommendations without requiring the model to be retrained.

The exported queue is intended to support the recommendations section of the final paper. Any figures generated during this notebook may also be reused to illustrate the distribution of recommendations and confidence levels.

In [39]:
from pathlib import Path
import json

# -------------------------------------------------------
# Export Directory
# -------------------------------------------------------

EXPORT_DIR = PROJECT_ROOT / "work" / "outputs"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------
# Export Ranked Queue
# -------------------------------------------------------

queue.sort_values("final_rank").to_csv(
    EXPORT_DIR / "refresh_queue_sample.csv",
    index=False
)

print("✓ refresh_queue_sample.csv exported.")

# -------------------------------------------------------
# Export Summary Statistics
# -------------------------------------------------------

summary = {
    "total_pages": int(len(queue)),
    "average_refresh_score": float(queue["final_refresh_score"].mean()),
    "average_model_probability": float(queue["best_model_probability"].mean()),
    "high_confidence": int((queue["confidence"] == "High").sum()),
    "medium_confidence": int((queue["confidence"] == "Medium").sum()),
    "low_confidence": int((queue["confidence"] == "Low").sum())
}

with open(EXPORT_DIR / "playbook_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

print("✓ playbook_summary.json exported.")

✓ refresh_queue_sample.csv exported.
✓ playbook_summary.json exported.


### Interpretation

The exported files provide a reproducible version of the final recommendation queue generated by the validated model. These files are intended to support documentation, reporting, and the recommendations section of the final research paper.

Because the notebook regenerates these outputs from the validated predictions, the workflow remains reproducible without manually editing exported files.

## Self-check

Before you submit, confirm each line honestly:

✅ Every section above is filled — markdown thinking AND the code that backs it

✅ The notebook runs top to bottom with no errors (Runtime → Run all)

✅ No client names, URLs, or private queries anywhere

✅ My claims use careful words: observed, measured, directional, decision-support

✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.